# can_servo — CAN-connected servo controller node

Spec: 12V input → TPS54302 buck (3.3V) → STM32G431CBUx MCU.  
AS5047D magnetic encoder via SPI for position feedback.  
TCAN330G CAN transceiver bridging MCU CANTX/RX to a 2-pin CAN bus connector.

All ICs use REAL KiCad library symbols (auto-resolved via MPN).

In [ ]:
import pathlib
import hw_toolkit as hw
from hw_toolkit.parts import Buck

board = hw.Board("can_servo")
board

In [ ]:
# 12V input connector (2-pin) — real KiCad Connector_Generic:Conn_01x02
j_vin = board.module(
    id="j_vin",
    category="connector",
    mpn="Conn_01x02",
    package="PinHeader_1x02_P2.54mm",
    lib_id="Connector_Generic:Conn_01x02",
    manufacturer="Generic",
    price_usd=0.10,
)
j_vin

In [ ]:
# 12V -> 3.3V buck (TPS54302, SOT-23-6) — full block with passives auto-wired
# Buck creates: buck_3v3_vin (12V), buck_3v3_vout (3.3V), gnd nets
buck = Buck(
    board,
    id="buck_3v3",
    mpn="TPS54302",
    package="SOT-23-6",
    vin=12.0,
    vout=3.3,
    l="10uH",
    cin="10uF",
    cout="22uF",
    cboot="100nF",
    rtop="31.6k",
    rbot="10k",
    manufacturer="Texas Instruments",
    price_usd=0.85,
)
buck

In [ ]:
# STM32G431CBUx MCU (UFQFPN-48) — resolves to MCU_ST_STM32G4:STM32G431CBUx
mcu = board.module(
    id="mcu",
    category="mcu",
    mpn="STM32G431CBUx",
    package="UFQFPN-48",
    manufacturer="STMicroelectronics",
    price_usd=3.20,
)
print(f"MCU lib_id: {mcu.lib_id}")
mcu

In [ ]:
# AS5047D magnetic rotary encoder (TSSOP-14) — resolves to Sensor_Magnetic:AS5047D
enc = board.module(
    id="enc",
    category="encoder",
    mpn="AS5047D",
    package="TSSOP-14",
    manufacturer="ams",
    price_usd=4.50,
)
print(f"Encoder lib_id: {enc.lib_id}")
enc

In [ ]:
# TCAN330G CAN transceiver (SOIC-8) — resolves to Interface_CAN_LIN:TCAN330G
can_ic = board.module(
    id="can_ic",
    category="can_transceiver",
    mpn="TCAN330G",
    package="SOIC-8",
    manufacturer="Texas Instruments",
    price_usd=1.20,
)
print(f"CAN IC lib_id: {can_ic.lib_id}")
can_ic

In [ ]:
# 2-pin CAN bus connector — real KiCad Connector_Generic:Conn_01x02
j_can = board.module(
    id="j_can",
    category="connector",
    mpn="Conn_01x02",
    package="PinHeader_1x02_P2.54mm",
    lib_id="Connector_Generic:Conn_01x02",
    manufacturer="Generic",
    price_usd=0.10,
)
j_can

In [ ]:
# 3-pin SWD debug header — real KiCad Connector_Generic:Conn_01x03
j_swd = board.module(
    id="j_swd",
    category="connector",
    mpn="Conn_01x03",
    package="PinHeader_1x03_P2.54mm",
    lib_id="Connector_Generic:Conn_01x03",
    manufacturer="Generic",
    price_usd=0.10,
)
j_swd

In [ ]:
# Decoupling caps (auto-resolve to Device:C)
c1 = board.capacitor("C1", "100nF", package="0402")   # MCU VDD bypass
c2 = board.capacitor("C2", "4.7uF", package="0805")   # MCU VDD bulk
c3 = board.capacitor("C3", "100nF", package="0402")   # MCU VDDA bypass
c4 = board.capacitor("C4", "100nF", package="0402")   # CAN transceiver VCC
c5 = board.capacitor("C5", "100nF", package="0402")   # MCU VREF+ decoupling

# SHDN pull-up resistor (TCAN330G SHDN: HIGH = normal operation)
r1 = board.resistor("R1", "10k", package="0402")

In [ ]:
# ============================================================
# Power nets
# Buck factory creates: gnd, buck_3v3_vin (12V), buck_3v3_vout (3.3V)
# ============================================================

gnd  = board.nets["gnd"]           # 0V common
v12  = board.nets["buck_3v3_vin"]  # 12V input (already has IC.VIN + Cin)
v3v3 = board.nets["buck_3v3_vout"] # 3.3V output (already has L.2 + Cout + Rtop)

# 12V input header → buck input
v12 += "j_vin.Pin_1"
gnd += "j_vin.Pin_2"

In [ ]:
# ============================================================
# 3.3V distribution
# ============================================================

# MCU power: VDD (3 power pins share the same name), VDDA, VBAT
v3v3 += "mcu.VDD", "mcu.VDDA", "mcu.VBAT"
v3v3 += "c1.1", "c2.1", "c3.1"

# CAN transceiver VCC
v3v3 += "can_ic.VCC", "c4.1"

# Encoder VDD
v3v3 += "enc.VDD"

# GND returns
gnd += "mcu.VSS"
gnd += "c1.2", "c2.2", "c3.2", "c4.2"
gnd += "can_ic.GND"
gnd += "enc.GND"

In [ ]:
# ============================================================
# VREF+ — filtered 3.3V analog reference
# ============================================================
vref = board.power("vref", voltage_v=3.3)
vref += "mcu.VREF+", "c5.1"
gnd  += "c5.2"

In [ ]:
# ============================================================
# SPI bus: MCU SPI1 <-> AS5047D encoder
# Pins: MOSI=PB5, MISO=PB4, SCK=PB3, ~CS=PA4 (GPIO)
# Declare nets manually (avoids unused bundled-CS problem from board.spi())
# ============================================================
spi_mosi = board.signal("spi_mosi", protocol="spi")
spi_miso = board.signal("spi_miso", protocol="spi")
spi_sck  = board.signal("spi_sck",  protocol="spi")
spi_cs   = board.signal("spi_cs",   protocol="spi")

spi_mosi += "mcu.PB5", "enc.MOSI"
spi_miso += "mcu.PB4", "enc.MISO"
spi_sck  += "mcu.PB3", "enc.CLK"
spi_cs   += "mcu.PA4", "enc.~{CS}"

In [ ]:
# ============================================================
# AS5047D unused pins: A/B/U/V/I·PWM/W·PWM → NC, TST → GND, VDD3V3 → NC
# ============================================================
gnd += "enc.TST"   # TST tied to GND per datasheet

nc_a    = board.nc("nc_enc_a");    nc_a    += "enc.A"
nc_b    = board.nc("nc_enc_b");    nc_b    += "enc.B"
nc_ipwm = board.nc("nc_enc_ipwm"); nc_ipwm += "enc.I/PWM"
nc_wpwm = board.nc("nc_enc_wpwm"); nc_wpwm += "enc.W/PWM"
nc_u    = board.nc("nc_enc_u");    nc_u    += "enc.U"
nc_v    = board.nc("nc_enc_v");    nc_v    += "enc.V"
nc_v33  = board.nc("nc_enc_vdd3"); nc_v33  += "enc.VDD3V3"

In [ ]:
# ============================================================
# CAN bus: MCU FDCAN1 → TCAN330G → 2-pin CAN connector
# Pins: CANTX=PA12 (FDCAN1_TX), CANRX=PA11 (FDCAN1_RX)
# ============================================================
can_tx = board.signal("can_tx", protocol="can")
can_rx = board.signal("can_rx", protocol="can")
canh   = board.signal("canh",   protocol="can")
canl   = board.signal("canl",   protocol="can")

can_tx += "mcu.PA12", "can_ic.TXD"
can_rx += "mcu.PA11", "can_ic.RXD"
canh   += "can_ic.CANH", "j_can.Pin_1"
canl   += "can_ic.CANL", "j_can.Pin_2"

# SHDN: pull HIGH to 3.3V via R1 → normal operation
can_shdn = board.power("can_shdn", voltage_v=3.3)
can_shdn += "can_ic.SHDN", "r1.1"
v3v3     += "r1.2"

# S pin: LOW → high-speed CAN mode
gnd += "can_ic.S"

In [ ]:
# ============================================================
# SWD debug interface: PA13=SWDIO, PA14=SWDCLK
# 3-pin connector: Pin_1=SWDIO, Pin_2=SWDCLK, Pin_3=GND
# ============================================================
swdio  = board.signal("swd_swdio",  protocol="swd")
swdclk = board.signal("swd_swdclk", protocol="swd")

swdio  += "mcu.PA13", "j_swd.Pin_1"
swdclk += "mcu.PA14", "j_swd.Pin_2"
gnd    += "j_swd.Pin_3"

In [ ]:
print(board.summary())

In [ ]:
# Inline render — see the placed + ELK-routed schematic
board.show()

In [ ]:
# PCB: footprints placed + ratsnest airwires (copper not routed yet).
res = board.write_pcb()
print("placed:", res.placed, "| skipped:", res.skipped)
board.show_pcb()

In [ ]:
# ERC gate: all ICs are real symbols → use ERC_REAL_SYMBOL_CODES
# Connectors (Connector_Generic:Conn_01x02/03) are also real KiCad symbols.
board.check_erc(expected_codes=hw.ERC_REAL_SYMBOL_CODES)

In [ ]:
out = pathlib.Path("/Users/juanantonioluera/ws/hw-toolkit/docs/projects/can_servo/can_servo.zip")
board.export_kicad(out, unzip=True, expected_codes=hw.ERC_REAL_SYMBOL_CODES)
print(f"Exported: {out}")